## LangCalc Inference

<img src="https://raw.githubusercontent.com/jedick/LangCalc/main/assets/langcalc-icon-outline.svg" alt="LangCalc icon" width="100">

[LangCalc](https://github.com/jedick/LangCalc) is a voice-driven calculator that uses a fine-tuned FunctionGemma model for function calling.
This notebook uses the fine-tuned model created with the [LangCalc Training](https://github.com/jedick/LangCalc/blob/main/model/notebooks/LangCalc-training.ipynb) notebook.

The inference example in this notebook only uses text input, and does not include speech transcription.

<a target="_blank" href="https://colab.research.google.com/github/jedick/LangCalc/blob/main/model/notebooks/LangCalc-inference.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>

<a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/model/notebooks/LangCalc-inference.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>

## Optional: Login to Hugging Face Hub

This step is not required to download the model, but you can login to HF Hub to enable higher rate limits.

In [ ]:
hf_login = False

if hf_login:
    """Log in to the Hugging Face Hub, using a Colab secret if available,
    otherwise falling back to an interactive prompt."""
    from huggingface_hub import login

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass

    if token:
        login(token)
        print("Logged in to Hugging Face Hub using the HF_TOKEN Colab secret.")
    else:
        login()

## Load model

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

model_id = "jedick/functiongemma-langcalc-en"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

## Define functions that our model can use

In [ ]:
def add(x: float, y: float):
    """
    Adds two numbers together (sum, total, plus).

    Args:
        x: the first number
        y: the second number

    Returns:
        result: the sum of x and y (x + y).
    """
    return {"result": x + y}

def subtract(x: float, y: float):
    """
    Subtracts one number from another (difference, minus).

    Args:
        x: the starting number
        y: the number to be subtracted from x

    Returns:
        result: the difference between x and y (x - y).
    """
    return {"result": x - y}

def multiply(x: float, y: float):
    """
    Multiplies two numbers together (product, times).

    Args:
        x: the first number
        y: the second number

    Returns:
        result: the product of x and y (x * y).
    """
    return {"result": x * y}

def divide(x: float, y: float):
    """
    Divides one number by another (quotient, over).

    Args:
        x: the numerator
        y: the denominator

    Returns:
        result: the quotient of x divided by y (x / y).
    """
    return {"result": x / y}

# The main interface

In [ ]:
def ask(prompt):
    """
    Runs calculation functions based on the prompt and returns a plain-language response

    Args:
        prompt: user's prompt

    Returns: text response
    """

    # Model's Turn

    tools = [add, subtract, multiply, divide]

    message = [
            # ESSENTIAL SYSTEM PROMPT:
            # This line activates the model's function calling logic.
            {"role": "developer", "content": "You are a model that can do function calling with the following functions"},
            {"role": "user", "content": prompt},
    ]
    print(message[0])
    print(f"Tools: {tools}")
    print(message[1])

    inputs = processor.apply_chat_template(message, tools=tools, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
    generated_tokens = out[0][len(inputs["input_ids"][0]):]
    output = processor.decode(generated_tokens, skip_special_tokens=True)

    # Developer's Turn
    # Note: Always validate function names and arguments before execution.

    import re

    def extract_tool_calls(text):
        def cast(v):
            try: return int(v)
            except:
                try: return float(v)
                except: return {'true': True, 'false': False}.get(v.lower(), v.strip("'\""))

        return [{
            "name": name,
            "arguments": {
                k: cast((v1 or v2).strip())
                for k, v1, v2 in re.findall(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]*))", args)
            }
        } for name, args in re.findall(r"<start_function_call>call:(\w+)\{(.*?)\}<end_function_call>", text, re.DOTALL)]

    calls = extract_tool_calls(output)
    if calls:
        message.append({
            "role": "assistant",
            "tool_calls": [{"type": "function", "function": call} for call in calls]
        })
        print(message[-1])

        # Call the function and get the result
        #####################################
        # WARNING: This is a demonstration. #
        #####################################
        # Using globals() to call functions dynamically can be dangerous in
        # production. In a real application, you should implement a secure way to
        # map function names to actual function calls, such as a predefined
        # dictionary of allowed tools and their implementations.
        results = [
            {"name": c['name'], "response": globals()[c['name']](**c['arguments'])}
            for c in calls
        ]

        message.append({
            "role": "tool",
            "content": results
        })
        print(message[-1])

    # Final Response
    # Finally, FunctionGemma reads the tool response and replies to the user.
    # NOTE: this is commented becuse the model at this point generally produces nonsense;
    # anyway, we are only interested in the numerical result, not a chatty summary of it.

    # inputs = processor.apply_chat_template(message, tools=tools, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    # out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
    # generated_tokens = out[0][len(inputs["input_ids"][0]):]
    # response = processor.decode(generated_tokens, skip_special_tokens=True)
    # message.append({"role": "assistant", "content": response})
    # print(message[-1])

In [ ]:
ask("subtract 10 from 1")

In [ ]:
ask("what's 21 times 20?")